# Stage 2 Notebook 30 - Exp2Y Fix lambda oscillation + extended training

**Why this exists.** While inspecting NB28 (Exp2W) the user noticed that the lane loss appears to oscillate epoch-by-epoch. Investigation pinpointed the cause: `grad_norm_calibration` recomputes `lambda = ||grad_det|| / ||grad_lane||` at step 1 of every epoch. Across Exp2W's 10 epochs:

| epoch | lambda_lane | total_loss |
|---|---:|---:|
| 3 | 0.500 | 3.95 |
| 4 | 0.500 | 3.83 |
| 5 | 0.500 | 3.83 |
| 6 | 1.851 | **7.97** ← jump |
| 7 | 2.000 | **8.63** ← max |
| 8 | 1.261 | 6.12 |

Lambda swings 4x between epochs (0.5 to 2.0 ceiling). When lambda grows, lane gets weighted more, lane gradients get larger, lane converges faster, grad_lane shrinks, NEXT epoch's ratio shoots up further -> runaway feedback loop. This is GradNorm's classic instability mode.

**The fix**: switch to `lambda_mode: fixed` with `lambda_lane: 1.0`. Eliminates the per-epoch recalibration entirely. Plus two related improvements:
- `geometry_warmup_epochs: 2 -> 1`: shrinks the warmup ramp jump that contributed ~1.0 to train_lane spike at epoch 1->2.
- `end_epoch: 10 -> 15`: with stable lambda, more training time should reward steadily.

Single-config-file change vs Exp2W (best so far at decoded_f1=0.043). Architecture identical.

Reference: Chen et al. 2018 'GradNorm' acknowledges instability and recommends EMA-smoothing; Kendall et al. 2018 'Multi-Task Learning Using Uncertainty' provides a theoretically grounded alternative -- tested separately as Exp2Z.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 15-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp25_rmt_gca_mask_fixed_lambda_long_joint_smoke.log
OK exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.0381 det_loss=3.1209 grad_cos=0.0412 lambda_lane=0.1102
  gate_stats={'gate/det_mean': 0.5023387670516968, 'gate/lane_mean': 0.5031609535217285, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short15'
    EPOCHS = 15
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15.tar --epochs 15 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp25_rmt_gca_mask_fixed_lambda_long_joint_short15_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp25_rmt_gca_mask_fixed_lambda_long_joint.yaml --curve-tar /content/d

0

## What to watch in Exp2Y training

Reference Exp2W (NB28, mask aux + grad_norm calibration): decoded_f1=0.0427, val_lane_f1=0.644, matched_iou=0.143, oracle_f1=0.074. lambda_lane oscillated between 0.5 and 2.0.

Pass criteria at epoch 15:
- **`lambda_lane` is 1.0 throughout** -- no oscillation. The smoke fix.
- **`val/lane/decoded_f1 >= 0.07`**: stable training + 1.5x more epochs should beat Exp2W's 0.043.
- **`val/matched_line_iou >= 0.20`**: geometry should accumulate progress without lambda noise.
- **`train_total` decreases monotonically** epoch by epoch (no sawtooth pattern from lambda jumps).

Failure signals:
- decoded_f1 stays at ~0.04: lambda oscillation wasn't the bottleneck; pivot to Exp2Z (uncertainty weighting) or Exp2AA (high resolution).
- Geometry diverges after extended training: fixed lambda lets lane dominate too much; reduce lambda_lane to 0.5.